In [5]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/playground-series-s6e4/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e4/train.csv
/kaggle/input/competitions/playground-series-s6e4/test.csv


In [6]:
# ================================
# 🚀 FULL PIPELINE (ONE CELL)
# ================================

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
import lightgbm as lgb
from catboost import CatBoostClassifier

print("🚀 Starting pipeline...")

# ================================
# LOAD DATA
# ================================
print("📂 Loading data...")

train = pd.read_csv('/kaggle/input/competitions/playground-series-s6e4/train.csv')
test = pd.read_csv('/kaggle/input/competitions/playground-series-s6e4/test.csv')

target = 'Irrigation_Need'

X = train.drop(columns=[target, 'id'])
y = train[target]

X_test = test.drop(columns=['id'])
test_ids = test['id']

print("✅ Data loaded:", X.shape, X_test.shape)

# ================================
# TARGET ENCODING
# ================================
le = LabelEncoder()
y = le.fit_transform(y)

print("✅ Target encoded")

# ================================
# FEATURE ENGINEERING
# ================================
print("⚙️ Creating features...")

def create_features(df):

    # Evapotranspiration proxy
    df['ET0'] = df['Temperature_C'] * df['Wind_Speed_kmh'] * df['Sunlight_Hours'] / (df['Humidity'] + 1)

    # Water balance
    df['water_balance'] = df['Rainfall_mm'] + df['Previous_Irrigation_mm'] - df['ET0']

    # Moisture stress
    df['moisture_stress'] = df['Temperature_C'] * (100 - df['Humidity']) / (df['Soil_Moisture'] + 1)

    # Soil retention
    df['soil_retention'] = df['Soil_Moisture'] * df['Organic_Carbon']

    # Salinity impact
    df['salinity_effect'] = df['Electrical_Conductivity'] * df['Soil_Moisture']

    # Interactions
    df['crop_stage'] = df['Crop_Type'] + "_" + df['Crop_Growth_Stage']
    df['region_season'] = df['Region'] + "_" + df['Season']

    # Ratios
    df['irrigation_per_area'] = df['Previous_Irrigation_mm'] / (df['Field_Area_hectare'] + 1)
    df['weather_intensity'] = df['Temperature_C'] * df['Sunlight_Hours'] / (df['Humidity'] + 1)
    df['rain_effectiveness'] = df['Rainfall_mm'] / (df['Temperature_C'] + 1)

    return df

X = create_features(X)
X_test = create_features(X_test)

print("✅ Feature engineering done")

# ================================
# ENCODE CATEGORICALS
# ================================
print("🔤 Encoding categoricals...")

cat_cols = X.select_dtypes(include='object').columns

for col in cat_cols:
    le_col = LabelEncoder()
    full = pd.concat([X[col], X_test[col]])
    le_col.fit(full)

    X[col] = le_col.transform(X[col])
    X_test[col] = le_col.transform(X_test[col])

print("✅ Encoding complete")

# ================================
# CV SETUP
# ================================
print("🔁 Setting up Stratified KFold...")

skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

oof_preds = np.zeros((len(X), 3))
test_preds_lgb = np.zeros((len(X_test), 3))
test_preds_cat = np.zeros((len(X_test), 3))

# ================================
# TRAIN LOOP
# ================================
print("🚀 Training models...")

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"\n🔁 Fold {fold+1}")

    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y[train_idx], y[val_idx]

    # LGBM
    print("🌲 Training LGBM...")
    lgb_model = lgb.LGBMClassifier(
        n_estimators=2000,
        learning_rate=0.03,
        num_leaves=64,
        min_child_samples=50,
        subsample=0.8,
        colsample_bytree=0.8,
        class_weight={0:1,1:1,2:5},
        random_state=42
    )

    lgb_model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        eval_metric='multi_logloss',
        callbacks=[lgb.early_stopping(100, verbose=False)]
    )

    oof_preds[val_idx] = lgb_model.predict_proba(X_val)
    test_preds_lgb += lgb_model.predict_proba(X_test) / 5

    print("✅ LGBM done")

    # CatBoost
    print("🐱 Training CatBoost...")
    cat_model = CatBoostClassifier(
        iterations=500,
        learning_rate=0.05,
        depth=8,
        loss_function='MultiClass',
        verbose=0
    )

    cat_model.fit(X_tr, y_tr)
    test_preds_cat += cat_model.predict_proba(X_test) / 5

    print("✅ CatBoost done")

print("\n✅ All folds complete")

# ================================
# CV SCORE
# ================================
oof_labels = np.argmax(oof_preds, axis=1)
cv_score = accuracy_score(y, oof_labels)

print(f"\n📊 CV Accuracy: {cv_score:.5f}")

# ================================
# ENSEMBLE
# ================================
print("🔗 Ensembling predictions...")

test_preds = 0.7 * test_preds_lgb + 0.3 * test_preds_cat

# ================================
# PSEUDO LABELING
# ================================
print("🧠 Applying pseudo-labeling...")

confidence = np.max(test_preds, axis=1)
pseudo_idx = confidence > 0.95

print(f"✅ Selected {pseudo_idx.sum()} pseudo samples")

X_pseudo = X_test[pseudo_idx]
y_pseudo = np.argmax(test_preds[pseudo_idx], axis=1)

X_aug = pd.concat([X, X_pseudo])
y_aug = np.concatenate([y, y_pseudo])

print("✅ Augmented dataset:", X_aug.shape)

# ================================
# FINAL TRAIN
# ================================
print("🚀 Training final model...")

final_model = lgb.LGBMClassifier(
    n_estimators=2000,
    learning_rate=0.03,
    num_leaves=64,
    min_child_samples=50,
    subsample=0.8,
    colsample_bytree=0.8,
    class_weight={0:1,1:1,2:5},
    random_state=42
)

final_model.fit(X_aug, y_aug)

# ================================
# FINAL PREDICTIONS
# ================================
print("📤 Generating submission...")

final_preds = final_model.predict(X_test)
final_preds = le.inverse_transform(final_preds)

submission = pd.DataFrame({
    'id': test_ids,
    'Irrigation_Need': final_preds
})

submission.to_csv('submission.csv', index=False)

print("🎉 DONE! Submission saved as submission.csv")

🚀 Starting pipeline...
📂 Loading data...
✅ Data loaded: (630000, 19) (270000, 19)
✅ Target encoded
⚙️ Creating features...
✅ Feature engineering done
🔤 Encoding categoricals...
✅ Encoding complete
🔁 Setting up Stratified KFold...
🚀 Training models...

🔁 Fold 1
🌲 Training LGBM...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.073350 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4777
[LightGBM] [Info] Number of data points in the train set: 420000, number of used features: 29
[LightGBM] [Info] Start training from score -4.324207
[LightGBM] [Info] Start training from score -1.455881
[LightGBM] [Info] Start training from score -0.282945
✅ LGBM done
🐱 Training CatBoost...
✅ CatBoost done

🔁 Fold 2
🌲 Training LGBM...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.072676 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bi